# Well-Known Regression Problems with Continuous Features

This notebook demonstrates how to work with popular regression datasets that have only continuous features.

## Datasets Covered:
1. **California Housing** - Best for high accuracy (R² ~ 0.80+)
2. **Diabetes** - Smaller dataset for quick experiments (R² ~ 0.45-0.50)

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_california_housing, load_diabetes
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score,
    mean_absolute_percentage_error
)

import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Set style for plots
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. California Housing Dataset (RECOMMENDED)

### Dataset Description:
- **Samples**: 20,640
- **Features**: 8 continuous features
- **Target**: Median house value (in $100,000s)
- **Expected R²**: 0.80+ with ensemble methods

### Features:
1. MedInc - median income
2. HouseAge - median house age
3. AveRooms - average rooms per household
4. AveBedrms - average bedrooms per household
5. Population - block population
6. AveOccup - average occupancy
7. Latitude - latitude
8. Longitude - longitude

In [ ]:
# Load California Housing Dataset
print("Loading California Housing dataset...")
california = fetch_california_housing(as_frame=True)

X = california.data
y = california.target

print(f"\nDataset shape: {X.shape}")
print(f"Features: {list(X.columns)}")
print(f"\nTarget statistics:")
print(y.describe())

In [ ]:
# Explore the data
print("First few rows:")
display(X.head())

print("\nData types:")
print(X.dtypes)

print("\nMissing values:")
print(X.isnull().sum())

In [ ]:
# Visualize feature distributions
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.ravel()

for idx, col in enumerate(X.columns):
    axes[idx].hist(X[col], bins=50, edgecolor='black', alpha=0.7)
    axes[idx].set_title(f'{col}')
    axes[idx].set_xlabel('Value')
    axes[idx].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

In [ ]:
# Visualize target distribution
plt.figure(figsize=(10, 5))
plt.hist(y, bins=50, edgecolor='black', alpha=0.7)
plt.title('Distribution of Median House Values')
plt.xlabel('House Value ($100k)')
plt.ylabel('Frequency')
plt.axvline(y.mean(), color='red', linestyle='--', label=f'Mean: ${y.mean():.2f}')
plt.legend()
plt.show()

In [ ]:
# Correlation matrix
plt.figure(figsize=(10, 8))
correlation_matrix = pd.concat([X, y.rename('Target')], axis=1).corr()
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

print("\nCorrelation with target:")
print(correlation_matrix['Target'].sort_values(ascending=False))

### Data Preprocessing

In [ ]:
# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

# Scale the features (important for linear models)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\nData preprocessed and ready for training!")

### Model Training and Evaluation

In [ ]:
def evaluate_model(model, X_train, X_test, y_train, y_test, model_name):
    """Train and evaluate a model."""
    # Train
    model.fit(X_train, y_train)
    
    # Predict
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)
    
    # Calculate metrics
    results = {
        'Model': model_name,
        'Train R²': r2_score(y_train, y_pred_train),
        'Test R²': r2_score(y_test, y_pred_test),
        'Train RMSE': np.sqrt(mean_squared_error(y_train, y_pred_train)),
        'Test RMSE': np.sqrt(mean_squared_error(y_test, y_pred_test)),
        'Train MAE': mean_absolute_error(y_train, y_pred_train),
        'Test MAE': mean_absolute_error(y_test, y_pred_test)
    }
    
    return results, y_pred_test

# Test multiple models
models = {
    'Ridge': (Ridge(alpha=1.0, random_state=RANDOM_STATE), True),  # (model, needs_scaling)
    'Lasso': (Lasso(alpha=1.0, random_state=RANDOM_STATE), True),
    'ElasticNet': (ElasticNet(alpha=1.0, random_state=RANDOM_STATE), True),
    'Random Forest': (RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1), False),
    'Gradient Boosting': (GradientBoostingRegressor(n_estimators=100, random_state=RANDOM_STATE), False)
}

results_list = []
predictions = {}

for name, (model, needs_scaling) in models.items():
    print(f"\nTraining {name}...")
    
    if needs_scaling:
        results, y_pred = evaluate_model(model, X_train_scaled, X_test_scaled, y_train, y_test, name)
    else:
        results, y_pred = evaluate_model(model, X_train, X_test, y_train, y_test, name)
    
    results_list.append(results)
    predictions[name] = y_pred
    
    print(f"  Train R²: {results['Train R²']:.4f}")
    print(f"  Test R²:  {results['Test R²']:.4f}")
    print(f"  Test RMSE: {results['Test RMSE']:.4f}")

In [ ]:
# Display results table
results_df = pd.DataFrame(results_list)
results_df = results_df.round(4)

print("\n" + "="*80)
print("MODEL COMPARISON - CALIFORNIA HOUSING DATASET")
print("="*80)
display(results_df)

# Find best model
best_model = results_df.loc[results_df['Test R²'].idxmax(), 'Model']
best_r2 = results_df['Test R²'].max()
print(f"\n🏆 Best Model: {best_model} (Test R² = {best_r2:.4f})")

In [ ]:
# Visualize predictions vs actual for best model
best_predictions = predictions[best_model]

plt.figure(figsize=(10, 6))
plt.scatter(y_test, best_predictions, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel('Actual Values ($100k)')
plt.ylabel('Predicted Values ($100k)')
plt.title(f'{best_model} - Predictions vs Actual Values')
plt.tight_layout()
plt.show()

# Residual plot
residuals = y_test - best_predictions
plt.figure(figsize=(10, 6))
plt.scatter(best_predictions, residuals, alpha=0.5)
plt.axhline(y=0, color='r', linestyle='--', lw=2)
plt.xlabel('Predicted Values ($100k)')
plt.ylabel('Residuals')
plt.title(f'{best_model} - Residual Plot')
plt.tight_layout()
plt.show()

## 2. Diabetes Dataset (Alternative)

### Dataset Description:
- **Samples**: 442
- **Features**: 10 continuous features (already standardized)
- **Target**: Disease progression measure
- **Expected R²**: 0.45-0.50

This is a smaller, more challenging dataset.

In [ ]:
# Load Diabetes Dataset
print("Loading Diabetes dataset...")
diabetes = load_diabetes(as_frame=True)

X_diab = diabetes.data
y_diab = diabetes.target

print(f"\nDataset shape: {X_diab.shape}")
print(f"Features: {list(X_diab.columns)}")
print(f"\nTarget statistics:")
print(y_diab.describe())

# Display first few rows
display(X_diab.head())

In [ ]:
# Split and train
X_train_diab, X_test_diab, y_train_diab, y_test_diab = train_test_split(
    X_diab, y_diab, test_size=0.2, random_state=RANDOM_STATE
)

# Train Ridge (works best for this dataset)
ridge_diab = Ridge(alpha=1.0, random_state=RANDOM_STATE)
results_diab, y_pred_diab = evaluate_model(
    ridge_diab, X_train_diab, X_test_diab, y_train_diab, y_test_diab, 'Ridge'
)

print("\n" + "="*60)
print("DIABETES DATASET RESULTS")
print("="*60)
for key, value in results_diab.items():
    if key != 'Model':
        print(f"{key}: {value:.4f}")

## Summary and Recommendations

### Best Choice: **California Housing Dataset**

**Why it's the best:**
1. ✅ **High Accuracy**: Test R² of 0.80+ achievable
2. ✅ **Large Dataset**: 20,640 samples for robust training
3. ✅ **All Continuous Features**: No categorical encoding needed
4. ✅ **Real-World Problem**: Practical house price prediction
5. ✅ **Well-Documented**: Widely used with known benchmarks

**Best Models:**
- **Random Forest**: R² ~ 0.80-0.81 (good balance)
- **Gradient Boosting**: R² ~ 0.78-0.80 (best performance)
- **Ridge Regression**: R² ~ 0.58 (fast, interpretable)

### Alternative: **Diabetes Dataset**
- Smaller dataset (442 samples)
- Best R² ~ 0.45-0.50
- Good for quick experiments and teaching
- Features already standardized

### Other Options (not shown here):
- **Wine Quality**: Predicting wine ratings (continuous target)
- **Bike Sharing**: Predicting bike rentals (temporal features)
- **Energy Efficiency**: Predicting heating/cooling loads